In [13]:
import geopandas as gpd
import pandas as pd
import py7zr, tempfile
import fiona
import shapely.geometry as sgeo
from tqdm.notebook import tqdm
import glob

In [14]:
network_path = "../../resources/network"
area_path = "../../results/spatial/iris.parquet"

output_path = "../../results/parking/network.parquet"

In [15]:
df_area = gpd.read_parquet(area_path)

In [16]:
df_geometry = []

for archive_path in glob.glob("{}/*".format(network_path)):
    with tempfile.TemporaryDirectory() as directory:
        with py7zr.SevenZipFile(archive_path) as archive:
                print("Extracting", archive_path)
                archive.extractall(directory) 

                source_path = list(glob.glob("{}/**/*.gpkg".format(directory), recursive = True))
                assert len(source_path) == 1
                source_path = source_path[0]

                print("  Processing", source_path)

                with fiona.open(source_path, layer = "troncon_de_route") as source:
                    for record in tqdm(source):
                        if pd.to_numeric(record["properties"]["importance"]) >= 3:
                            df_geometry.append(sgeo.LineString(record["geometry"]["coordinates"]))            

df_geometry = gpd.GeoDataFrame(pd.DataFrame({ "geometry": df_geometry }), crs = "EPSG:2154")

Extracting ../../resources/network\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D001_2022-03-15.7z
  Processing C:\Users\LEBESC~1\AppData\Local\Temp\tmpy3mepa1y\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D001_2022-03-15\BDTOPO\1_DONNEES_LIVRAISON_2022-03-00088\BDT_3-0_GPKG_LAMB93_D001-ED2022-03-15\BDT_3-0_GPKG_LAMB93_D001-ED2022-03-15.gpkg


  0%|          | 0/320366 [00:00<?, ?it/s]

Extracting ../../resources/network\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D038_2022-03-15.7z
  Processing C:\Users\LEBESC~1\AppData\Local\Temp\tmp0az_v3i9\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D038_2022-03-15\BDTOPO\1_DONNEES_LIVRAISON_2022-03-00088\BDT_3-0_GPKG_LAMB93_D038-ED2022-03-15\BDT_3-0_GPKG_LAMB93_D038-ED2022-03-15.gpkg


  0%|          | 0/414377 [00:00<?, ?it/s]

Extracting ../../resources/network\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D042_2022-03-15.7z
  Processing C:\Users\LEBESC~1\AppData\Local\Temp\tmpxbw8gcxn\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D042_2022-03-15\BDTOPO\1_DONNEES_LIVRAISON_2022-03-00088\BDT_3-0_GPKG_LAMB93_D042-ED2022-03-15\BDT_3-0_GPKG_LAMB93_D042-ED2022-03-15.gpkg


  0%|          | 0/274522 [00:00<?, ?it/s]

Extracting ../../resources/network\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D069_2022-03-15.7z
  Processing C:\Users\LEBESC~1\AppData\Local\Temp\tmp0coctt5j\BDTOPO_3-0_TOUSTHEMES_GPKG_LAMB93_D069_2022-03-15\BDTOPO\1_DONNEES_LIVRAISON_2022-03-00088\BDT_3-0_GPKG_LAMB93_D069-ED2022-03-15\BDT_3-0_GPKG_LAMB93_D069-ED2022-03-15.gpkg


  0%|          | 0/320226 [00:00<?, ?it/s]

In [17]:
df_network = []

for index, record in tqdm(df_area.iterrows(), total = len(df_area)):
    indices = df_geometry.sindex.intersection(record["geometry"].bounds)
    df_clip = df_geometry.iloc[indices].clip(record["geometry"])
    df_network.append({ "iris": record["iris"], "length": df_clip["geometry"].length.sum() })
    
df_network = pd.DataFrame.from_records(df_network)

  0%|          | 0/2472 [00:00<?, ?it/s]

In [18]:
df_network.to_parquet(output_path)